# 03. Baseline Regression Modeling

완성된 `data/processed/transactions.csv`를 수정하지 않고, 수치형 feature 기반 아파트 거래 단가 예측 baseline을 학습합니다.

- 비교 정책: 중개거래만 사용하는 Policy A, 중개거래와 unknown을 함께 사용하는 Policy B
- 비교 모델: LinearRegression, Ridge, Keras MLP
- 실행 방식: `RUN_MODE = "smoke"`로 빠른 검증 후 `RUN_MODE = "full"`로 전체 학습

## 1. 환경 및 패키지 확인

이 노트북은 `Python (tf)` 커널 기준입니다. TensorFlow/Keras가 없는 환경에서는 선형 모델 셀까지는 확인할 수 있지만 MLP 학습은 실행되지 않습니다.

In [1]:
from pathlib import Path
import inspect
import json
import math
import platform
import random
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from tensorflow import keras

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("tensorflow:", tf.__version__)
print("keras:", getattr(keras, "__version__", "unknown"))

Python: 3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ]
Python executable: /opt/anaconda3/envs/tf/bin/python
Platform: macOS-26.5.1-arm64-arm-64bit
pandas: 3.0.3
numpy: 2.4.4
scikit-learn: 1.8.0
tensorflow: 2.21.0
keras: 3.14.0


## 2. 경로 및 실행 설정

`RUN_MODE`만 바꾸면 같은 코드로 smoke run과 full run을 수행합니다.

In [2]:
# 1) 경로와 실행 모드 설정
# repo root 또는 final_project 안에서 실행해도 같은 기준 경로를 사용합니다.
current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

DATA_PATH = PROJECT_DIR / "data" / "processed" / "transactions.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# smoke는 빠른 검증용, full은 전체 학습용입니다.
RUN_MODE = "smoke"  # "smoke" or "full"
RANDOM_STATE = 42

SMOKE_LIMITS = {
    "train": 200_000,
    "valid": 50_000,
    "test": 50_000,
    "recent_holdout": 50_000,
}

MLP_BATCH_SIZE = 8192
MLP_MAX_EPOCHS = 20
MLP_PATIENCE = 3

assert RUN_MODE in {"smoke", "full"}, "RUN_MODE must be either 'smoke' or 'full'."
assert DATA_PATH.exists(), f"transactions.csv not found: {DATA_PATH}"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("RUN_MODE:", RUN_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)

PROJECT_DIR: /Users/gwongwangjae/goorm-ai-language-course/final_project
DATA_PATH: /Users/gwongwangjae/goorm-ai-language-course/final_project/data/processed/transactions.csv
RUN_MODE: smoke
OUTPUT_DIR: /Users/gwongwangjae/goorm-ai-language-course/final_project/outputs
MODEL_DIR: /Users/gwongwangjae/goorm-ai-language-course/final_project/models


## 3. CSV 로드

모델링에 필요한 컬럼만 `usecols`로 읽습니다. 입력 CSV는 재생성하거나 수정하지 않습니다.

In [3]:
# 2) 데이터 로드
# 모델링에 필요한 컬럼만 읽어 메모리 사용량을 줄입니다.
USECOLS = [
    "transaction_id",
    "area_m2",
    "floor",
    "age_years",
    "deal_date",
    "trade_type",
    "is_cancelled",
    "price_total",
    "price_per_m2",
    "target",
    "complex_prev_price_per_m2",
    "complex_prev_missing",
    "prev_deal_gap_days",
]

DTYPES = {
    "transaction_id": "string",
    "area_m2": "float32",
    "floor": "float32",
    "age_years": "float32",
    "trade_type": "string",
    "is_cancelled": "Int8",
    "price_total": "float32",
    "price_per_m2": "float32",
    "target": "float32",
    "complex_prev_price_per_m2": "float32",
    "complex_prev_missing": "Int8",
    "prev_deal_gap_days": "float32",
}

df = pd.read_csv(DATA_PATH, usecols=USECOLS, dtype=DTYPES, parse_dates=["deal_date"])
print(df.shape)
display(df.head())

(3593663, 13)


,transaction_id,area_m2,floor,age_years,deal_date,trade_type,is_cancelled,price_total,price_per_m2,target,complex_prev_price_per_m2,complex_prev_missing,prev_deal_gap_days
0,2019-01-01_11170-38_7e36a63d,84.779999,3.0,25.0,2019-01-01,unknown,0,109000.0,1285.680542,7.159043,1633.640015,0,111.0
1,2019-01-01_11290-3622_d8f212f5,84.970001,4.0,11.0,2019-01-01,unknown,0,52000.0,611.980713,6.416701,706.131592,0,77.0
2,2019-01-01_11350-156_9051c591,114.989998,2.0,19.0,2019-01-01,unknown,0,65200.0,567.005798,6.340370,547.873718,0,129.0
3,2019-01-01_11410-4768_19cff35a,48.480000,6.0,1.0,2019-01-01,unknown,0,42000.0,866.336609,6.764274,NaN,1,NaN
4,2019-01-01_11500-192_b7cc1f6c,51.029999,9.0,25.0,2019-01-01,unknown,0,40500.0,793.650818,6.676643,783.852661,0,63.0


## 4. Feature 생성 및 검증

정답값이나 미래/식별자/거래 유형 컬럼은 feature에서 제외합니다. `complex_prev_price_per_m2`는 원본값을 직접 쓰지 않고 로그 변환값만 사용합니다.

In [4]:
# 3) Feature 목록과 leakage 방지
# 정답/식별자/날짜/거래상태 컬럼은 모델 입력에서 제외합니다.
NUMERIC_FEATURES = [
    "area_m2",
    "floor",
    "is_basement_floor",
    "age_years",
    "log_complex_prev_price_per_m2",
    "complex_prev_missing",
    "prev_deal_gap_months",
]
CATEGORICAL_FEATURES = ["prev_deal_gap_bucket"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

HARD_LEAKAGE_COLUMNS = {
    "target",
    "price_total",
    "price_per_m2",
    "reported_at",
    "deal_date",
    "deal_ym",
    "transaction_id",
    "raw_complex_name",
    "normalized_complex_name",
    "complex_id",
    "legal_dong_code",
    "sgg_code",
    "trade_type",
    "is_cancelled",
    "prev_price_ratio",
    "complex_prev_price_per_m2",
    "prev_deal_gap_days",
}
assert not (set(FEATURES) & HARD_LEAKAGE_COLUMNS), set(FEATURES) & HARD_LEAKAGE_COLUMNS


# 4) 모델링용 파생 feature 생성
def add_features(input_df: pd.DataFrame) -> pd.DataFrame:
    out = input_df.copy()
    out["target"] = np.log(out["price_per_m2"].astype("float64"))
    out["is_basement_floor"] = (out["floor"] < 0).astype("float32")
    prev_price = out["complex_prev_price_per_m2"].astype("float64")
    out["log_complex_prev_price_per_m2"] = np.where(prev_price > 0, np.log(prev_price), np.nan)
    out["complex_prev_missing"] = out["complex_prev_missing"].fillna(1).astype("float32")
    out["prev_deal_gap_months"] = out["prev_deal_gap_days"].astype("float64") / 30.4375

    gap = out["prev_deal_gap_days"]
    bucket = pd.Series("missing", index=out.index, dtype="string")
    bucket[(gap >= 0) & (gap <= 30)] = "0-30"
    bucket[(gap >= 31) & (gap <= 90)] = "31-90"
    bucket[(gap >= 91) & (gap <= 180)] = "91-180"
    bucket[(gap >= 181) & (gap <= 365)] = "181-365"
    bucket[gap >= 366] = "366+"
    out["prev_deal_gap_bucket"] = bucket.fillna("missing")
    return out


df = add_features(df)
assert df["target"].notna().all(), "target contains null values."
assert np.isfinite(df["target"]).all(), "target contains non-finite values."

display(df[FEATURES + ["target"]].head())

,area_m2,floor,is_basement_floor,age_years,log_complex_prev_price_per_m2,complex_prev_missing,prev_deal_gap_months,prev_deal_gap_bucket,target
0,84.779999,3.0,0.0,25.0,7.398566,0.0,3.646817,91-180,7.159043
1,84.970001,4.0,0.0,11.0,6.559802,0.0,2.529774,31-90,6.416701
2,114.989998,2.0,0.0,19.0,6.306045,0.0,4.238193,91-180,6.340370
3,48.480000,6.0,0.0,1.0,NaN,1.0,NaN,missing,6.764274
4,51.029999,9.0,0.0,25.0,6.664221,0.0,2.069815,31-90,6.676644


## 5. 정책, Split, Smoke Sampling

시간 기준 split은 고정합니다.

- train: `deal_date <= 2023-12-31`
- valid: `2024-01-01 ~ 2024-12-31`
- test: `2025-01-01 ~ 2025-12-31`
- recent_holdout: `2026-01-01` 이후

In [5]:
# 5) 학습 정책과 시간 기준 split
# Policy A/B는 trade_type 포함 범위만 다르고 나머지 조건은 동일합니다.
POLICIES = {
    "policy_a_broker_only": {
        "label": "Policy A: is_cancelled == 0 and trade_type == 중개거래",
        "trade_types": ["중개거래"],
    },
    "policy_b_broker_unknown": {
        "label": "Policy B: is_cancelled == 0 and trade_type in [중개거래, unknown]",
        "trade_types": ["중개거래", "unknown"],
    },
}

SPLIT_ORDER = ["train", "valid", "test", "recent_holdout"]


def policy_filter(input_df: pd.DataFrame, trade_types: list[str]) -> pd.Series:
    trade_type = input_df["trade_type"].fillna("unknown")
    return (input_df["is_cancelled"] == 0) & trade_type.isin(trade_types)


def split_frames(policy_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    splits = {
        "train": policy_df.loc[policy_df["deal_date"] <= "2023-12-31"],
        "valid": policy_df.loc[(policy_df["deal_date"] >= "2024-01-01") & (policy_df["deal_date"] <= "2024-12-31")],
        "test": policy_df.loc[(policy_df["deal_date"] >= "2025-01-01") & (policy_df["deal_date"] <= "2025-12-31")],
        "recent_holdout": policy_df.loc[policy_df["deal_date"] >= "2026-01-01"],
    }
    for split_name, split_df in splits.items():
        assert len(split_df) > 0, f"{split_name} split is empty."
    return splits


# 6) smoke 모드 샘플링
def apply_smoke_sampling(splits: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    if RUN_MODE != "smoke":
        return splits
    sampled = {}
    for split_name, split_df in splits.items():
        limit = SMOKE_LIMITS[split_name]
        if len(split_df) > limit:
            sampled[split_name] = split_df.sample(n=limit, random_state=RANDOM_STATE).sort_values("deal_date")
        else:
            sampled[split_name] = split_df.copy()
    return sampled


policy_data = {}
count_rows = []
for policy_name, policy_cfg in POLICIES.items():
    filtered = df.loc[policy_filter(df, policy_cfg["trade_types"])].copy()
    full_splits = split_frames(filtered)
    run_splits = apply_smoke_sampling(full_splits)
    policy_data[policy_name] = {
        "config": policy_cfg,
        "full_splits": full_splits,
        "run_splits": run_splits,
    }
    for split_name in SPLIT_ORDER:
        count_rows.append({
            "policy": policy_name,
            "policy_label": policy_cfg["label"],
            "split": split_name,
            "full_rows": len(full_splits[split_name]),
            "run_rows": len(run_splits[split_name]),
        })

counts_df = pd.DataFrame(count_rows)
display(counts_df)

,policy,policy_label,split,full_rows,run_rows
0,policy_a_broker_only,Policy A: is_cancelled == 0 and trade_type == ...,train,574245,200000
1,policy_a_broker_only,Policy A: is_cancelled == 0 and trade_type == ...,valid,384432,50000
2,policy_a_broker_only,Policy A: is_cancelled == 0 and trade_type == ...,test,442008,50000
3,policy_a_broker_only,Policy A: is_cancelled == 0 and trade_type == ...,recent_holdout,209468,50000
4,policy_b_broker_unknown,Policy B: is_cancelled == 0 and trade_type in ...,train,2342081,200000
5,policy_b_broker_unknown,Policy B: is_cancelled == 0 and trade_type in ...,valid,384432,50000
6,policy_b_broker_unknown,Policy B: is_cancelled == 0 and trade_type in ...,test,442008,50000
7,policy_b_broker_unknown,Policy B: is_cancelled == 0 and trade_type in ...,recent_holdout,209468,50000


## 6. Preprocessing 및 모델 정의

Imputer, scaler, encoder는 각 policy의 train split에만 fit합니다.

In [6]:
# 7) 전처리와 모델 정의
# sklearn의 ColumnTransformer는 딥러닝 Transformer가 아니라 컬럼별 전처리 도구입니다.
# 전처리기는 train split에만 fit하고 다른 split에는 transform만 적용합니다.
def make_one_hot_encoder() -> OneHotEncoder:
    params = {"handle_unknown": "ignore"}
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        params["sparse_output"] = False
    else:
        params["sparse"] = False
    return OneHotEncoder(**params)


def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        sparse_threshold=0.0,
    )


def build_mlp(input_dim: int) -> keras.Model:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"],
    )
    return model


def xy(split_df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    X = split_df[FEATURES].copy()
    y = split_df["target"].to_numpy(dtype="float64")
    assert np.isfinite(y).all(), "target contains non-finite values."
    return X, y

## 7. 평가 함수

로그 단가 예측값을 `exp`로 복원해 단가와 총액 기준 오류를 함께 계산합니다. `price_total` 단위가 CSV의 총액 단위이므로 `total_price_mae_manwon`도 같은 단위입니다.

In [7]:
# 8) 평가 지표 계산
# 로그 예측값을 다시 가격 스케일로 복원해 단가/총액 오류를 함께 봅니다.
def evaluate_predictions(
    split_df: pd.DataFrame,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    policy_name: str,
    model_name: str,
    split_name: str,
) -> dict[str, float | str | int]:
    y_pred = np.asarray(y_pred, dtype="float64").reshape(-1)
    assert len(y_true) == len(y_pred) == len(split_df)
    pred_price_per_m2 = np.exp(y_pred)
    actual_price_per_m2 = split_df["price_per_m2"].to_numpy(dtype="float64")
    pred_total = pred_price_per_m2 * split_df["area_m2"].to_numpy(dtype="float64")
    actual_total = split_df["price_total"].to_numpy(dtype="float64")
    assert (pred_price_per_m2 > 0).all(), "exp(pred) must be positive."

    valid_mape_mask = actual_price_per_m2 > 0
    return {
        "run_mode": RUN_MODE,
        "policy": policy_name,
        "model": model_name,
        "split": split_name,
        "rows": len(split_df),
        "log_mae": mean_absolute_error(y_true, y_pred),
        "log_rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
        "price_per_m2_mae": mean_absolute_error(actual_price_per_m2, pred_price_per_m2),
        "price_per_m2_mape": np.mean(np.abs((actual_price_per_m2[valid_mape_mask] - pred_price_per_m2[valid_mape_mask]) / actual_price_per_m2[valid_mape_mask])),
        "total_price_mae_manwon": mean_absolute_error(actual_total, pred_total),
    }


def prediction_sample(split_df: pd.DataFrame, y_pred: np.ndarray, policy_name: str, model_name: str, split_name: str, n: int = 200) -> pd.DataFrame:
    sample = split_df[["transaction_id", "deal_date", "area_m2", "price_total", "price_per_m2", "target"]].copy()
    sample["policy"] = policy_name
    sample["model"] = model_name
    sample["split"] = split_name
    sample["pred_target"] = np.asarray(y_pred, dtype="float64").reshape(-1)
    sample["pred_price_per_m2"] = np.exp(sample["pred_target"])
    sample["pred_total"] = sample["pred_price_per_m2"] * sample["area_m2"].astype("float64")
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=RANDOM_STATE)
    return sample.sort_values(["policy", "model", "split", "deal_date"])

## 8. 모델 학습 및 비교

Policy A/B 각각에서 LinearRegression, Ridge, Keras MLP를 학습하고 모든 split에 대해 같은 지표를 계산합니다.

In [8]:
# 9) 모델 학습과 비교
# 각 policy마다 LinearRegression, Ridge, Keras MLP를 같은 split으로 평가합니다.
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

metrics_rows = []
prediction_samples = []
best_mlp = None

for policy_name, payload in policy_data.items():
    print(f"\n=== {policy_name}: {payload['config']['label']} ===")
    splits = payload["run_splits"]
    X_train, y_train = xy(splits["train"])

    sklearn_models = {
        "linear_regression": LinearRegression(),
        "ridge_alpha_1": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    }

    for model_name, estimator in sklearn_models.items():
        print(f"Training {model_name}...")
        pipeline = Pipeline([
            ("preprocessor", build_preprocessor()),
            ("model", estimator),
        ])
        pipeline.fit(X_train, y_train)
        for split_name in SPLIT_ORDER:
            X_split, y_split = xy(splits[split_name])
            pred = pipeline.predict(X_split)
            metrics_rows.append(evaluate_predictions(splits[split_name], y_split, pred, policy_name, model_name, split_name))
            if split_name in {"valid", "test", "recent_holdout"}:
                prediction_samples.append(prediction_sample(splits[split_name], pred, policy_name, model_name, split_name, n=100))

    # MLP는 별도 preprocessor로 배열을 만든 뒤 early stopping을 적용합니다.
    print("Training keras_mlp...")
    mlp_preprocessor = build_preprocessor()
    X_train_mlp = mlp_preprocessor.fit_transform(X_train).astype("float32")
    X_valid_mlp, y_valid = xy(splits["valid"])
    X_valid_mlp = mlp_preprocessor.transform(X_valid_mlp).astype("float32")

    mlp = build_mlp(X_train_mlp.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=MLP_PATIENCE,
            restore_best_weights=True,
        )
    ]
    history = mlp.fit(
        X_train_mlp,
        y_train,
        validation_data=(X_valid_mlp, y_valid),
        epochs=MLP_MAX_EPOCHS,
        batch_size=MLP_BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    valid_pred_for_selection = mlp.predict(X_valid_mlp, batch_size=MLP_BATCH_SIZE, verbose=0).reshape(-1)
    valid_log_mae = mean_absolute_error(y_valid, valid_pred_for_selection)
    if best_mlp is None or valid_log_mae < best_mlp["valid_log_mae"]:
        best_mlp = {
            "policy": policy_name,
            "valid_log_mae": valid_log_mae,
            "model": mlp,
            "preprocessor": mlp_preprocessor,
            "history": history.history,
        }

    for split_name in SPLIT_ORDER:
        X_split, y_split = xy(splits[split_name])
        X_split_mlp = mlp_preprocessor.transform(X_split).astype("float32")
        pred = mlp.predict(X_split_mlp, batch_size=MLP_BATCH_SIZE, verbose=0).reshape(-1)
        metrics_rows.append(evaluate_predictions(splits[split_name], y_split, pred, policy_name, "keras_mlp", split_name))
        if split_name in {"valid", "test", "recent_holdout"}:
            prediction_samples.append(prediction_sample(splits[split_name], pred, policy_name, "keras_mlp", split_name, n=100))

metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df.sort_values(["policy", "model", "split"]).reset_index(drop=True)
predictions_sample_df = pd.concat(prediction_samples, ignore_index=True)

display(metrics_df)


=== policy_a_broker_only: Policy A: is_cancelled == 0 and trade_type == 중개거래 ===
Training linear_regression...
Training ridge_alpha_1...
Training keras_mlp...
Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 29.3951 - mae: 5.3433 - val_loss: 24.0001 - val_mae: 4.8256
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.8738 - mae: 3.8499 - val_loss: 9.3978 - val_mae: 2.9450
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.9914 - mae: 1.9487 - val_loss: 2.0149 - val_mae: 1.1004
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6101 - mae: 0.9250 - val_loss: 1.3394 - val_mae: 0.8343
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.0988 - mae: 0.7810 - val_loss: 0.9810 - val_mae: 0.7358
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8526 - mae: 0.6962 - val_loss: 0.7961 - val_mae: 0.6532
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6680 - mae: 0.6110 - val_loss: 0.6322 - val_mae: 0.5771
Epoch 8/20
25/25 ━━━━━━━━

,run_mode,policy,model,split,rows,log_mae,log_rmse,price_per_m2_mae,price_per_m2_mape,total_price_mae_manwon
0,smoke,policy_a_broker_only,keras_mlp,recent_holdout,50000,0.139000,0.194845,100.831612,0.143779,8373.756382
1,smoke,policy_a_broker_only,keras_mlp,test,50000,0.136790,0.188930,107.478967,0.140401,9252.339595
2,smoke,policy_a_broker_only,keras_mlp,train,200000,0.135089,0.186053,70.954152,0.138320,5839.433387
3,smoke,policy_a_broker_only,keras_mlp,valid,50000,0.132505,0.182188,90.812192,0.135145,7881.070401
4,smoke,policy_a_broker_only,linear_regression,recent_holdout,50000,0.077475,0.120304,48.569969,0.078517,3657.401046
5,smoke,policy_a_broker_only,linear_regression,test,50000,0.074678,0.114400,49.176797,0.074268,3881.383770
6,smoke,policy_a_broker_only,linear_regression,train,200000,0.084097,0.128946,38.269066,0.085476,2922.816474
7,smoke,policy_a_broker_only,linear_regression,valid,50000,0.073671,0.111944,41.657350,0.074147,3291.370441
8,smoke,policy_a_broker_only,ridge_alpha_1,recent_holdout,50000,0.077476,0.120304,48.571939,0.078518,3657.542531
9,smoke,policy_a_broker_only,ridge_alpha_1,test,50000,0.074679,0.114401,49.179114,0.074270,3881.558000


## 9. 결과 저장

Smoke run에서도 metrics, summary, prediction sample을 저장합니다. Full run에서는 valid 기준 가장 좋은 MLP와 해당 preprocessor를 추가로 저장합니다.

In [ ]:
# 10) 결과 저장
# smoke는 결과 CSV/MD만, full은 best MLP artifact까지 저장합니다.
METRICS_PATH = OUTPUT_DIR / "baseline_regression_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "baseline_regression_summary.md"
PREDICTIONS_SAMPLE_PATH = OUTPUT_DIR / "baseline_predictions_sample.csv"
BEST_MODEL_PATH = MODEL_DIR / "baseline_mlp_best.keras"
BEST_PREPROCESSOR_PATH = MODEL_DIR / "baseline_preprocessor_best.joblib"
RUN_CONFIG_PATH = MODEL_DIR / "baseline_run_config.json"


def df_to_markdown(table_df: pd.DataFrame, floatfmt: str | None = None) -> str:
    formatted = table_df.copy()
    if floatfmt is not None:
        for col in formatted.select_dtypes(include=["float", "float32", "float64"]).columns:
            formatted[col] = formatted[col].map(lambda value: format(value, floatfmt) if pd.notna(value) else "")
    formatted = formatted.astype("string").fillna("")
    headers = list(formatted.columns)
    rows = formatted.values.tolist()
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(lines)


metrics_df.to_csv(METRICS_PATH, index=False)
predictions_sample_df.to_csv(PREDICTIONS_SAMPLE_PATH, index=False)

valid_metrics = metrics_df.loc[metrics_df["split"] == "valid"].copy()
best_valid = valid_metrics.sort_values("log_mae").iloc[0]

mlp_valid = valid_metrics.loc[valid_metrics["model"] == "keras_mlp"].sort_values("log_mae")
best_mlp_valid_row = mlp_valid.iloc[0]

# valid 기준 best model, recent 성능 변화, 선형 대비 MLP 개선 여부를 summary에 정리합니다.
comparison_rows = []
for policy_name in POLICIES:
    policy_valid = valid_metrics.loc[valid_metrics["policy"] == policy_name]
    linear_valid = policy_valid.loc[policy_valid["model"] == "linear_regression", "log_mae"].iloc[0]
    mlp_valid_value = policy_valid.loc[policy_valid["model"] == "keras_mlp", "log_mae"].iloc[0]
    comparison_rows.append({
        "policy": policy_name,
        "linear_valid_log_mae": linear_valid,
        "mlp_valid_log_mae": mlp_valid_value,
        "mlp_improvement_vs_linear": linear_valid - mlp_valid_value,
        "mlp_improved": mlp_valid_value < linear_valid,
    })
comparison_df = pd.DataFrame(comparison_rows)

test_recent_rows = []
for _, row in valid_metrics[["policy", "model"]].drop_duplicates().iterrows():
    subset = metrics_df.loc[(metrics_df["policy"] == row["policy"]) & (metrics_df["model"] == row["model"])]
    test_log_mae = subset.loc[subset["split"] == "test", "log_mae"].iloc[0]
    recent_log_mae = subset.loc[subset["split"] == "recent_holdout", "log_mae"].iloc[0]
    test_recent_rows.append({
        "policy": row["policy"],
        "model": row["model"],
        "test_log_mae": test_log_mae,
        "recent_log_mae": recent_log_mae,
        "recent_minus_test_log_mae": recent_log_mae - test_log_mae,
    })
test_recent_df = pd.DataFrame(test_recent_rows)

summary_lines = []
summary_lines.append("# Baseline Regression Summary")
summary_lines.append("")
summary_lines.append(f"- run_mode: `{RUN_MODE}`")
summary_lines.append(f"- random_state: `{RANDOM_STATE}`")
summary_lines.append(f"- input: `{DATA_PATH}`")
summary_lines.append("")
summary_lines.append("## Policy Row Counts")
summary_lines.append(df_to_markdown(counts_df))
summary_lines.append("")
summary_lines.append("## Metrics")
summary_lines.append(df_to_markdown(metrics_df, floatfmt=".6f"))
summary_lines.append("")
summary_lines.append("## Best Model by Valid log_mae")
summary_lines.append(f"- policy: `{best_valid['policy']}`")
summary_lines.append(f"- model: `{best_valid['model']}`")
summary_lines.append(f"- valid_log_mae: `{best_valid['log_mae']:.6f}`")
summary_lines.append("")
summary_lines.append("## Best MLP by Valid log_mae")
summary_lines.append(f"- policy: `{best_mlp_valid_row['policy']}`")
summary_lines.append(f"- valid_log_mae: `{best_mlp_valid_row['log_mae']:.6f}`")
summary_lines.append("")
summary_lines.append("## Test vs Recent Holdout Difference")
summary_lines.append(df_to_markdown(test_recent_df, floatfmt=".6f"))
summary_lines.append("")
summary_lines.append("## MLP Improvement vs Linear Regression")
summary_lines.append(df_to_markdown(comparison_df, floatfmt=".6f"))
summary_lines.append("")
summary_lines.append("## Output Files")
summary_lines.append(f"- metrics: `{METRICS_PATH}`")
summary_lines.append(f"- summary: `{SUMMARY_PATH}`")
summary_lines.append(f"- prediction_sample: `{PREDICTIONS_SAMPLE_PATH}`")

SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

if RUN_MODE == "full":
    assert best_mlp is not None, "No MLP candidate was trained."
    best_mlp["model"].save(BEST_MODEL_PATH)
    joblib.dump(best_mlp["preprocessor"], BEST_PREPROCESSOR_PATH)
    run_config = {
        "run_mode": RUN_MODE,
        "random_state": RANDOM_STATE,
        "features": FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "target": "ln(price_per_m2)",
        "best_mlp_policy": best_mlp["policy"],
        "best_mlp_valid_log_mae": float(best_mlp["valid_log_mae"]),
        "mlp_batch_size": MLP_BATCH_SIZE,
        "mlp_max_epochs": MLP_MAX_EPOCHS,
        "mlp_patience": MLP_PATIENCE,
        "policy_counts": counts_df.to_dict(orient="records"),
    }
    RUN_CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved best MLP:", BEST_MODEL_PATH)
    print("Saved preprocessor:", BEST_PREPROCESSOR_PATH)
    print("Saved run config:", RUN_CONFIG_PATH)
else:
    print("Smoke run: model artifacts are not saved. Set RUN_MODE = 'full' to save best MLP artifacts.")

for path in [METRICS_PATH, SUMMARY_PATH, PREDICTIONS_SAMPLE_PATH]:
    assert path.exists(), f"Expected output file was not created: {path}"

print("Saved metrics:", METRICS_PATH)
print("Saved summary:", SUMMARY_PATH)
print("Saved prediction sample:", PREDICTIONS_SAMPLE_PATH)

## 10. 다음 단계 판단 기준

- valid/test/recent에서 MLP가 선형 모델보다 안정적으로 개선되는지 확인합니다.
- recent_holdout 성능이 test보다 크게 나빠지면 2026년 분포 변화나 직전거래 feature 품질을 추가 점검합니다.
- 다음 단계에서는 단지/지역 범주형 feature 또는 embedding을 추가할지 결정합니다.